# 02: Growth-Induced Pressure (GIP) Analysis

## Overview
This notebook applies the image analysis pipeline to time-lapse chamber images of confined *E. coli* growth, quantifying how bacterial proliferation generates mechanical pressure (GIP) over time.

### Pipeline Stages
1. **X-axis analysis:** All 5 chambers — measures horizontal wall separation over time.
2. **Y-axis analysis:** Chamber-1 only — verifies isotropic deformation assumption (αy ≈ 1).
3. **GIP calculation:** Converts horizontal strain to pressure using the effective Young's modulus.
4. **Output:** GIP vs Time plots for all 5 chambers; CSV export.

**Key parameters (Report Section 2.2):**
- Pixel size: 0.06587 µm/px | Frame interval: 30 min
- Eeff = 4.337 MPa | Lx₀ = 31.21 µm (Chamber-1 initial width)
- Crop region (x-axis): y = [100, 300] | Min wall area: 700 px²

## 1. Imports & Configuration

In [ ]:
import os
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from skimage import measure
from scipy.ndimage import binary_closing, binary_dilation
from skimage.morphology import disk
from natsort import natsorted

# ── Paths ─────────────────────────────────────────────────────────────────
# BUG FIX [B]: replaced hardcoded absolute paths with generic relative paths
base_dir      = os.path.dirname(os.getcwd())
chambers_root = os.path.join(base_dir, "data", "processed", "masks", "chambers")
output_root   = os.path.join(base_dir, "data", "results")
output_csv_x  = os.path.join(output_root, "Chamber_Deformation_Xaxis.csv")
output_csv_y  = os.path.join(output_root, "Chamber1_Deformation_Yaxis.csv")
output_csv_gip= os.path.join(output_root, "Chamber_GIP_vs_Time.csv")
plot_folder   = os.path.join(output_root, "wall_verification", "chambers")

os.makedirs(output_root, exist_ok=True)
os.makedirs(plot_folder, exist_ok=True)

# ── Physical constants ────────────────────────────────────────────────────
pixel_size    = 0.06587    # µm/pixel
frame_interval= 0.5        # hours (30-min interval)
num_points    = 1000       # resampling resolution for chamber walls
min_wall_area = 700        # px² — minimum region size to be considered a wall
crop_y_start  = 100        # px — middle-region crop for x-axis analysis
crop_y_end    = 300
crop_x_start  = 100        # px — left-region crop for y-axis analysis
crop_x_end    = 200

# GIP parameters (Report Section 2.2 — verified values)
Eeff  = 4.337e6   # Pa (4.337 MPa)
Lx0   = 31.21     # µm — Chamber-1 initial horizontal width

chamber_names = [f"Chamber-{i}" for i in range(1, 6)]
print("Configuration loaded.")

## 2. Helper Functions

In [ ]:
def resample_points(coords, num_points):
    """Linearly resample wall coordinates to num_points equal-spacing points."""
    dists = np.sqrt(np.sum(np.diff(coords, axis=0)**2, axis=1))
    cum   = np.insert(np.cumsum(dists), 0, 0)
    tgt   = np.linspace(0, cum[-1], num_points)
    return np.stack([np.interp(tgt, cum, coords[:, 0]),
                     np.interp(tgt, cum, coords[:, 1])], axis=1)


def preprocess_and_crop(image_array, y_start=None, y_end=None,
                         x_start=None, x_end=None):
    """
    Apply morphological operations then zero-mask everything outside the crop region.
    Accepts either a y-crop (x-axis analysis) or x-crop (y-axis analysis).
    BUG FIX [L]: Gemini removed all morphological preprocessing; restored here.
    """
    struct  = disk(3)
    closed  = binary_closing(image_array > 0, structure=struct)
    dilated = binary_dilation(closed,  structure=struct)
    cropped = np.zeros_like(dilated, dtype=bool)
    if y_start is not None:
        cropped[y_start:y_end, :] = dilated[y_start:y_end, :]
    elif x_start is not None:
        cropped[:, x_start:x_end] = dilated[:, x_start:x_end]
    return cropped


print("Helper functions defined.")

## 3. X-Axis Analysis — All 5 Chambers

Measures horizontal wall separation at each time-point by cropping y = [100, 300],
identifying the two lateral walls, resampling, and computing mean |x₁ − x₂|.

**BUG FIX [K]:** Gemini used centroid-to-centroid distance — explicitly rejected in Report
Section 2.1.1 as unreliable for irregular/non-parallel walls. Restored resampled-point method.

In [ ]:
xaxis_results = []

for chamber in chamber_names:
    cpath = os.path.join(chambers_root, chamber, "image_files")
    if not os.path.exists(cpath):
        print(f"WARNING: {cpath} not found — skipping.")
        continue

    files = natsorted([f for f in os.listdir(cpath) if f.endswith('.tif')])
    x_dists, x_stds = [], []

    ch_plot_dir = os.path.join(plot_folder, chamber, "xaxis")
    os.makedirs(ch_plot_dir, exist_ok=True)

    for i, fname in enumerate(files):
        # BUG FIX [G]: restored mask inversion (Fiji exports inverted)
        arr = 255 - np.array(Image.open(os.path.join(cpath, fname)))

        processed = preprocess_and_crop(arr,
                        y_start=crop_y_start, y_end=crop_y_end)  # BUG FIX [L]

        labeled  = measure.label(processed, connectivity=2)
        regions  = [r for r in measure.regionprops(labeled)
                    if r.area >= min_wall_area]
        if len(regions) < 2:
            print(f"  WARN: <2 walls in {fname} — skipping frame.")
            continue

        # Sort by x-centroid: left wall and right wall
        regions = sorted(regions, key=lambda r: r.centroid[1])[:2]
        # BUG FIX [K]: use resampled wall coords, not centroid distance
        w1 = resample_points(np.array(regions[0].coords), num_points)
        w2 = resample_points(np.array(regions[1].coords), num_points)
        x_dist = np.mean(np.abs(w1[:, 1] - w2[:, 1])) * pixel_size
        x_std  = np.std( np.abs(w1[:, 1] - w2[:, 1])) * pixel_size
        x_dists.append(x_dist)
        x_stds.append(x_std)

        time_h = i * frame_interval
        defo   = x_dist - x_dists[0]
        xaxis_results.append({
            "Chamber": chamber, "Frame": i+1, "Time_h": time_h,
            "Distance_um": x_dist, "Std_um": x_std, "Deformation_um": defo
        })

        # Save verification plot
        fig, ax = plt.subplots(figsize=(5, 3))
        ax.imshow(arr, cmap='gray')
        ax.plot(w1[:, 1], w1[:, 0], 'r-', lw=0.7, label='Wall L')
        ax.plot(w2[:, 1], w2[:, 0], 'b-', lw=0.7, label='Wall R')
        ax.set_title(f"{chamber} x-axis — {fname}", fontsize=7)
        ax.axis('off'); ax.legend(fontsize=6)
        fig.savefig(os.path.join(ch_plot_dir, fname.replace('.tif','_x.png')),
                    dpi=120, bbox_inches='tight')
        plt.close(fig)

    print(f"{chamber}: {len(x_dists)} frames processed.")

    # Per-chamber deformation plot
    times = [r['Time_h'] for r in xaxis_results if r['Chamber']==chamber]
    dists = [r['Distance_um'] for r in xaxis_results if r['Chamber']==chamber]
    stds  = [r['Std_um'] for r in xaxis_results if r['Chamber']==chamber]
    plt.figure(figsize=(8,4))
    plt.errorbar(times, dists, yerr=stds, fmt='o-', capsize=3)
    plt.xlabel('Time (h)'); plt.ylabel('Wall Separation (µm)')
    plt.title(f'Wall Separation Over Time — {chamber}')
    plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

# BUG FIX [N]: Gemini had no CSV export
df_x = pd.DataFrame(xaxis_results)
df_x.to_csv(output_csv_x, index=False)
print(f"\nX-axis results saved to: {output_csv_x}")

## 4. Y-Axis Analysis — Chamber-1 (Isotropic Deformation Verification)

Crops the left region (x = [100, 200]) to isolate the top and bottom walls, excluding
the vertical side walls and inlet. Measures vertical separation over time.

Report result: αy = ΔLy/ΔLx = 0.98, confirming near-isotropic deformation.

**BUG FIX [M]:** Gemini dropped this section entirely; restored from original.

In [ ]:
cpath = os.path.join(chambers_root, "Chamber-1", "image_files")
files = natsorted([f for f in os.listdir(cpath) if f.endswith('.tif')])

y_dists, y_stds, yaxis_results = [], [], []
ch1_plot_dir = os.path.join(plot_folder, "Chamber-1", "yaxis")
os.makedirs(ch1_plot_dir, exist_ok=True)

for i, fname in enumerate(files):
    arr = 255 - np.array(Image.open(os.path.join(cpath, fname)))

    processed = preprocess_and_crop(arr,
                    x_start=crop_x_start, x_end=crop_x_end)

    labeled = measure.label(processed, connectivity=2)
    regions = [r for r in measure.regionprops(labeled)
               if r.area >= min_wall_area]
    if len(regions) < 2:
        print(f"  WARN: <2 walls in {fname} — skipping.")
        continue

    # Sort by y-centroid: top and bottom walls
    regions = sorted(regions, key=lambda r: r.centroid[0])[:2]
    w3 = resample_points(np.array(regions[0].coords), num_points)
    w4 = resample_points(np.array(regions[1].coords), num_points)
    y_dist = np.mean(np.abs(w3[:, 0] - w4[:, 0])) * pixel_size
    y_std  = np.std( np.abs(w3[:, 0] - w4[:, 0])) * pixel_size
    y_dists.append(y_dist)
    y_stds.append(y_std)

    time_h = i * frame_interval
    defo   = y_dist - y_dists[0]
    # BUG FIX [P]: original had label='Chamber-5' here — copy-paste error; corrected to Chamber-1
    yaxis_results.append({
        "Chamber": "Chamber-1", "Frame": i+1, "Time_h": time_h,
        "Y_Distance_um": y_dist, "Y_Std_um": y_std, "Y_Deformation_um": defo
    })

    fig, ax = plt.subplots(figsize=(5, 3))
    ax.imshow(arr, cmap='gray')
    ax.plot(w3[:, 1], w3[:, 0], 'g-', lw=0.7, label='Wall Top')
    ax.plot(w4[:, 1], w4[:, 0], 'y-', lw=0.7, label='Wall Bottom')
    ax.set_title(f"Chamber-1 y-axis — {fname}", fontsize=7)
    ax.axis('off'); ax.legend(fontsize=6)
    fig.savefig(os.path.join(ch1_plot_dir, fname.replace('.tif','_y.png')),
                dpi=120, bbox_inches='tight')
    plt.close(fig)

df_y = pd.DataFrame(yaxis_results)
df_y.to_csv(output_csv_y, index=False)

# Overlay x vs y deformation for Chamber-1
df_x1 = df_x[df_x['Chamber']=='Chamber-1'].copy()
times_x = df_x1['Time_h'].values
defo_x  = df_x1['Deformation_um'].values
times_y = df_y['Time_h'].values
defo_y  = df_y['Y_Deformation_um'].values

plt.figure(figsize=(9, 4))
plt.plot(times_x, defo_x, 'o-', label='X-axis deformation', color='steelblue')
plt.plot(times_y, defo_y, 's--', label='Y-axis deformation', color='tomato')
plt.xlabel('Time (h)'); plt.ylabel('Deformation (µm)')
plt.title('Chamber-1: X vs Y Deformation — Isotropic Verification (αy ≈ 0.98)')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print(f"Y-axis results saved to: {output_csv_y}")

## 5. Growth-Induced Pressure (GIP) vs Time

GIP is computed from horizontal strain multiplied by the effective Young's modulus:

$$\text{GIP} = E_{\text{eff}} \cdot \frac{\Delta L_x}{L_{x,0}}$$

where ΔLx = current distance − baseline distance, Lx₀ = initial chamber width.

**BUG FIX [O]:** Neither the original nor the Gemini notebook computed GIP in pressure units.
This cell completes the stated pipeline goal (Report Section 3.3, Figures 2–6).

Verified parameters: Eeff = 4.337 MPa, Lx₀ = 31.21 µm (Chamber-1, t=0).

In [ ]:
gip_results = []

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes_flat  = axes.flatten()

for idx, chamber in enumerate(chamber_names):
    sub = df_x[df_x['Chamber'] == chamber]
    if sub.empty:
        axes_flat[idx].set_visible(False)
        continue

    # Horizontal strain: ΔLx / Lx0
    delta_Lx = sub['Deformation_um'].values          # µm
    strain   = delta_Lx / Lx0                        # dimensionless

    # GIP = Eeff × strain  (Pa → kPa)
    gip_kpa  = Eeff * strain / 1e3                   # kPa
    times    = sub['Time_h'].values

    for t, s, g in zip(times, strain, gip_kpa):
        gip_results.append({"Chamber": chamber, "Time_h": t,
                             "Strain": s, "GIP_kPa": g})

    axes_flat[idx].plot(times, gip_kpa, 'o-', markersize=3)
    axes_flat[idx].set_xlabel('Time (h)', fontsize=10)
    axes_flat[idx].set_ylabel('GIP (kPa)', fontsize=10)
    axes_flat[idx].set_title(f'GIP Over Time — {chamber}', fontsize=10)
    axes_flat[idx].grid(alpha=0.3)

axes_flat[-1].set_visible(False)  # 6th panel unused
plt.suptitle('Growth-Induced Pressure vs Time (All Chambers)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(output_root, 'GIP_vs_Time_all_chambers.png'), dpi=300)
plt.show()

df_gip = pd.DataFrame(gip_results)
df_gip.to_csv(output_csv_gip, index=False)
print(f"GIP results saved to: {output_csv_gip}")
df_gip.groupby('Chamber')['GIP_kPa'].describe().round(2)